# HippocampalPlaceCellExample

This notebook demonstrates a clean-room place-cell decoding workflow in 2D.

Workflow:
1. Simulate a 2D trajectory and Gaussian place fields.
2. Generate Poisson spike counts from place fields.
3. Decode position using state-index estimators.
4. Validate normalized position error.

Reference: DOI `10.1016/j.jneumeth.2012.08.009`, PMID `22981419`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nstat.decoding import DecodingAlgorithms

rng = np.random.default_rng(2026)
print('Seed:', 2026)


In [ ]:
n_units = 30
n_time = 320
grid_side = 14
grid = np.linspace(0.0, 1.0, grid_side)
gx, gy = np.meshgrid(grid, grid, indexing='xy')
state_xy = np.column_stack([gx.ravel(), gy.ravel()])
n_states = state_xy.shape[0]

traj = np.zeros((n_time, 2), dtype=float)
traj[0] = np.array([0.5, 0.5])
velocity = np.zeros(2, dtype=float)
for t in range(1, n_time):
    velocity = 0.85 * velocity + 0.12 * rng.normal(size=2)
    traj[t] = np.clip(traj[t - 1] + velocity, 0.0, 1.0)

d2 = np.sum((state_xy[None, :, :] - traj[:, None, :]) ** 2, axis=2)
true_state = np.argmin(d2, axis=1)

centers = rng.uniform(0.0, 1.0, size=(n_units, 2))
sigma = 0.18
dist2 = np.sum((state_xy[None, :, :] - centers[:, None, :]) ** 2, axis=2)
tuning = 0.03 + 0.80 * np.exp(-0.5 * dist2 / (sigma ** 2))

spike_counts = np.zeros((n_units, n_time), dtype=float)
for t in range(n_time):
    spike_counts[:, t] = rng.poisson(tuning[:, true_state[t]])

decoded_wc = DecodingAlgorithms.decode_weighted_center(spike_counts, tuning)
decoded_wc = np.clip(np.rint(decoded_wc), 0, n_states - 1).astype(int)

log_tuning = np.log(np.clip(tuning, 1e-12, None))
decoded_ml = np.zeros(n_time, dtype=int)
for t in range(n_time):
    k = spike_counts[:, t][:, None]
    ll = np.sum(k * log_tuning - tuning, axis=0)
    decoded_ml[t] = int(np.argmax(ll))

xy_true = state_xy[true_state]
xy_wc = state_xy[decoded_wc]
xy_ml = state_xy[decoded_ml]

rmse_wc = float(np.sqrt(np.mean(np.sum((xy_wc - xy_true) ** 2, axis=1))))
rmse_ml = float(np.sqrt(np.mean(np.sum((xy_ml - xy_true) ** 2, axis=1))))
rmse_norm = rmse_ml / np.sqrt(2.0)

print('RMSE weighted-center:', rmse_wc)
print('RMSE ML decoder:', rmse_ml)
print('Normalized RMSE (ML):', rmse_norm)

assert rmse_ml <= rmse_wc + 1e-8, 'ML decoder should be at least as good as weighted-center.'
assert rmse_norm < 0.30, f'Place decoding error too high: {rmse_norm}'


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

axes[0].plot(xy_true[:, 0], xy_true[:, 1], label='true', linewidth=1.3)
axes[0].plot(xy_ml[:, 0], xy_ml[:, 1], label='decoded ML', linewidth=1.0, alpha=0.85)
axes[0].set_title('2D trajectory decoding')
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
axes[0].set_aspect('equal', adjustable='box')
axes[0].legend(loc='upper right')

example_unit = 8
im = axes[1].imshow(
    tuning[example_unit].reshape(grid_side, grid_side),
    origin='lower',
    extent=[0, 1, 0, 1],
    cmap='jet',
    aspect='equal',
)
axes[1].set_title(f'Example place field (unit {example_unit})')
axes[1].set_xlabel('x')
axes[1].set_ylabel('y')
fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()
